# Fracture Caption Evaluation

Reproduces every results table in the paper from **`configs/eval.yml`**, which lists the
human references and one entry per prediction CSV (your model and each ablation):

| Section | Paper table |
|---|---|
| 3. Summary (headline) | `tab:conditioning_ablation` — one row per prediction set |
| 4. CLIP metrics | `tab:clip-metrics` — CLIPScore, Ref-Text, RefCLIP, R@k |
| 5. Caption collapse | `tab:collapse-stats` + distinct-n, near-duplicates, intra-class similarity |
| 6. CIDEr-R bootstrap | `tab:cider-bootstrap` — paired bootstrap, Bonferroni |
| 7. Fracture type | BOUNDARY / INTERIOR accuracy vs the references |
| 8. Human scores | optional, 0–5 scores per predicted caption |

**Use:** set `CONFIG` in the **Configuration** cell, then *Run All*.

**Add an ablation:** run `scripts/infer_mae.py` for it, then add one entry to `configs/eval.yml`:
```yaml
predictions:
  point_only:
    csv: outputs/predictions_point_only.csv
    label: Point-only
    run_config: configs/mae_point_only.yml   # optional: settings shown in the provenance table
```
`enabled: false` keeps an entry without evaluating it. Unknown keys are an error.
The computation lives in `eval_captions.py` (same folder), which also runs headless:
`python evaluation/eval_captions.py --config configs/eval.yml` (add `--only NAME` / `--pred NAME=CSV`).
Paths are relative to the repository root; the setup cell switches there automatically.
A run config such as `configs/mae_v3.yml` also works as `CONFIG` (its `infer.out` becomes the only set).

**Definitions worth knowing**
* **CLIPScore** is reported as `100·cos(image, caption)` (`clipscore_report: cos100`), which is
  what the paper's 23.x values are. RefCLIP = harmonic mean of `2.5·cos` and Ref-Text (Hessel et al.).
* **Prediction *k*** always means `caption_idx = k−1`. Collapse and CIDEr-R use one sample,
  `primary_pred`. The submitted tables used `caption_pred_1` (= Prediction 2) for those —
  set `primary_pred: 1` to reproduce them exactly.
* **CIDEr-R baseline** `symmetric` averages ref1→ref2 and ref2→ref1; `legacy` = ref1→ref2 only (old notebook).
* **R@k ties** (identical captions) are scored at their expected value under random tie-breaking
  (`r_precision_ties: expected`), so collapsed captions earn no credit from index order.
  `index` reproduces the old `torch.topk` behaviour.
* Evaluation runs on the objects covered by **every** prediction set, so variants are comparable.

## 0. Setup

In [ ]:
# %pip install torch transformers pandas numpy pillow pyyaml tqdm matplotlib --quiet
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# Work from the repository root, wherever Jupyter was started (paths in configs/ are root-relative)
REPO = Path.cwd().resolve()
while not (REPO / "configs").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)
for p in (REPO / "evaluation", REPO / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print("Repository root:", REPO)

import eval_captions as ec
importlib.reload(ec)            # pick up edits to eval_captions.py without restarting

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

## 1. Configuration

`CONFIG` is the eval file. `OVERRIDES` changes `metrics` / `references` keys for this
session only (same names as in the file). `ONLY` restricts to some prediction sets.

```python
OVERRIDES = {"primary_pred": 1, "cider_baseline": "legacy",
             "r_precision_ties": "index"}        # → submitted-paper numbers
OVERRIDES = {"n_bootstrap": 2000}                # quick look
ONLY      = ["curvature_disk", "point_only"]     # subset of eval.yml's predictions
```

In [ ]:
CONFIG    = "configs/eval.yml"
OVERRIDES = {}
ONLY      = None          # e.g. ["curvature_disk", "point_only"]

S = ec.load_settings(CONFIG, OVERRIDES)
if ONLY:
    entries = {k: v for k, v in ec.prediction_entries(S).items() if k in ONLY}
    missing = set(ONLY) - set(entries)
    assert not missing, f"ONLY: unknown prediction set(s) {sorted(missing)}"
    S = ec.load_settings(CONFIG, {**OVERRIDES, "predictions": entries})
print(ec.describe_settings(S))

## 2. Load and merge references + predictions

In [ ]:
data = ec.load_data(S)

print(f"Objects evaluated : {len(data.objects)}  ({data.objects['class'].nunique()} classes)")
print(f"References        : {data.true_cols}")
for name, cols in data.pred_cols.items():
    print(f"Predictions       : {name}: {cols}")
for note in data.notes:
    print("Note:", note)
display(data.coverage)
display(data.objects["class"].value_counts().rename("objects").to_frame().T)

# Provenance: which run config produced each prediction set
variants = ec.variants_table(S)
display(variants.dropna(axis=1, how="all"))
for w in variants.get("warning", pd.Series(dtype=object)).dropna():
    print("WARNING:", w)

## 3. Compute all metrics

Loads CLIP once and embeds the overview image of each object plus every caption.
Embeddings are cached in `<output_dir>/.clip_cache` (`embed_cache: true`), so re-runs
after changing only statistical settings are fast.

In [ ]:
embedder = ec.CLIPEmbedder(
    S["clip_model"], S["device"], S["img_batch_size"], S["txt_batch_size"],
    Path(S["output_dir"]) / ".clip_cache" if S["embed_cache"] else None,
)
print(f"CLIP {S['clip_model']} on {embedder.device}")

R = ec.run(S, embedder=embedder, data=data)
for note in R.notes:
    print("Note:", note)

## 4. Summary table

One row per prediction set, plus the human reference row — the layout of the ablation table.

| Column | Meaning | Better |
|---|---|---|
| Unique | share of captions that appear exactly once (case-insensitive) | ↑ |
| Top-1 | share taken by the most frequent caption | ↓ |
| NearDup | share whose nearest same-class caption has CLIP text cos ≥ `near_dup_threshold` | ↓ |
| IntraSim | mean pairwise CLIP text cosine within a class (object-weighted) | ↓ |
| ΔCIDEr-R | CIDEr-R(pred vs refs) − CIDEr-R(ref vs other ref), with bootstrap CI | ≈ 0 |
| CLIPScore / Ref-Text / RefCLIP / R@k | CLIP-based agreement (Section 5) | ↑ |
| FracAcc | BOUNDARY/INTERIOR accuracy where the references agree | ↑ |
| ParseErr | share of samples that failed to parse as JSON | ↓ |

In [ ]:
tables = ec.summary_tables(R)
display(Markdown(f"**Primary sample for collapse / CIDEr-R: Prediction {S['primary_pred'] + 1} "
                 f"(caption_idx {S['primary_pred']})**"))
display(tables["headline"])

Full text summary (also written to `summary.txt`):

In [ ]:
print(ec.summary_text(R))

## 5. CLIP-based metrics

Ref-Text / RefCLIP for the references are the human-vs-human baseline (each reference
scored against the other), shown once, as in the paper.

In [ ]:
print(f"Chance R@1 over a pool of {R.n_with_image}: {R.clip.attrs['chance_R@1']:.5f}")
display(tables["clip"])

In [ ]:
# Per-class R@k (harder, collapse-sensitive: same-class templated captions collide)
k1 = S["r_precision_k"][0]
rp = R.r_precision_class.pivot_table(index="class", columns=["set", "caption"], values=f"R@{k1}")
display(rp.round(4))

## 6. Caption collapse per class

In [ ]:
for name in R.collapse:
    display(Markdown(f"**{ec.label(S, name)}** — Prediction {S['primary_pred'] + 1}"))
    display(tables[f"collapse:{name}"])
display(Markdown("**Human references (Reference 1)** — the diversity ceiling"))
display(ec._fmt(R.collapse_refs))

## 7. CIDEr-R gap vs the human baseline (paired bootstrap)

In [ ]:
for name, t in R.cider.items():
    display(Markdown(f"**{ec.label(S, name)}** — gap = pred − baseline; {S['n_bootstrap']} resamples; "
                     f"Bonferroni α = {t.attrs['alpha_bonferroni']:.5f}"))
    display(tables[f"cider:{name}"])

## 8. Fracture type (BOUNDARY / INTERIOR)

In [ ]:
if "fracture" in tables:
    display(tables["fracture"])
else:
    print("No fracture-type labels found in references or predictions.")

## 9. Human scores (optional)

Add `human_scores: <scored csv>` (the predictions CSV with a 0–5 `score` per caption) to a
prediction entry in `configs/eval.yml`. Spearman ρ shows which automatic metric tracks human judgement.

In [ ]:
if len(R.human):
    display(tables["human"])
    display(R.human_by_class.round(3))
else:
    print("No human scores configured (predictions.<name>.human_scores in configs/eval.yml).")

## 10. Save tables (CSV + LaTeX)

In [ ]:
for f in ec.save(R):
    print(" -", f)

## 11. Qualitative check — lowest-RefCLIP predictions

Shows the images, the references and the prediction for the worst cases of each prediction set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def show_example(row, image_root=S["image_root"], max_images=6):
    images = ec.parse_list(row["images_used"])[:max_images]
    n = max(len(images), 1)
    cols = min(3, n)
    fig, axes = plt.subplots(-(-n // cols), cols, figsize=(3.6 * cols, 3.2 * -(-n // cols)))
    for i, ax in enumerate(np.atleast_1d(axes).ravel()):
        ax.set_axis_off()
        if i < len(images):
            try:
                ax.imshow(Image.open(Path(image_root) / images[i]).convert("RGB"))
                ax.set_title(Path(images[i]).name, fontsize=7)
            except Exception:
                ax.text(0.5, 0.5, f"[missing]\n{Path(images[i]).name}", ha="center", va="center", fontsize=7)
    fig.suptitle(f"[{ec.label(S, row['set'])}] {row['class']} — RefCLIP {row['refclip']:.3f}", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("PRED :", row["prediction"])
    for c in [c for c in row.index if c.startswith("ref_")]:
        print(f"{c.upper():<5}:", row[c])
    print(row["path"])


worst = ec.worst_examples(R, S["n_examples"])
for _, row in worst.iterrows():
    show_example(row)